## Agent-scoped memory
Give the agent its own persistent identity that evolves over time. Agent-scoped memory is shared across all users, so the agent builds up its own persona, accumulated knowledge, and learned preferences through every conversation. As it interacts with users, it develops expertise, refines its approach, and remembers what works. It can also learn and update skills when it has write access.


In [1]:
from dotenv import load_dotenv
import os 
load_dotenv()
os.environ["OPENROUTER_API_KEY"] = os.getenv("OPENROUTER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import (
    CompositeBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.store.memory import InMemoryStore


# ============================================================
# 1. LONG-TERM MEMORY STORE
# ============================================================

store = InMemoryStore()


# ============================================================
# 2. BACKEND CONFIGURATION
# ============================================================
backend = CompositeBackend(
    default=StateBackend(),

    routes={
        "/memories/": StoreBackend(
            namespace=("my-assistant",),
        ),

        "/skills/": StoreBackend(
            namespace=("my-assistant",),
        ),
    },
)
# ============================================================
# 3. CREATE DEEP AGENT
# ============================================================

agent = create_deep_agent(
    model="openrouter:openai/gpt-4o-mini",

    # Project-level memory/instructions
    memory=[
        r"projects\agent.md"
    ],

    # Skills
    skills=[
        r"skills\python\SKILL.md"
    ],

    backend=backend,

    # IMPORTANT:
    # Give the agent the long-term store
    store=store,
)


In [15]:
config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My name is Nayyab."
            }
        ]
    },
    config=config,
)



In [16]:

result1 = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Who m i?."
            }
        ]
    },
    config=config,
)



In [17]:
result


{'messages': [HumanMessage(content='My name is Nayyab.', additional_kwargs={}, response_metadata={}, id='aa9bb277-c7ed-4232-be13-8c3a22852e7c'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model_name': 'openai/gpt-4o-mini', 'id': 'gen-1791286063-czmwXmlEqlSqOre1XmhU', 'created': 1791286063, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 0.0002844, 'cost_details': {'upstream_inference_completions_cost': 1.68e-05, 'upstream_inference_prompt_cost': 0.0002676, 'upstream_inference_cost': 0.0002844}, 'system_fingerprint': 'fp_d0d66bff01'}, id='lc_run--01a110f8-3099-7972-8099-d6da5ef9337f-0', tool_calls=[{'name': 'edit_file', 'args': {'file_path': 'user_info.txt', 'content': "User's name is Nayyab."}, 'id': 'call_yc8eFok6i6Rjk0eOqGOyGgFK', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 3512, 'output_tokens': 28, 'total_tokens': 3540, 'input_token_details': {'cache_read':

### UserScope Memory
Give each user their own memory file. The agent remembers preferences, context, and history per user while core agent instructions stay fixed. Users can also have per-user skills if stored in a user-scoped backend.


In [19]:
from deepagents import create_deep_agent
from deepagents.backends import (
    CompositeBackend,
    StateBackend,
    StoreBackend,
)
from langgraph.store.memory import InMemoryStore


# ============================================================
# 1. LONG-TERM STORE
# ============================================================

store = InMemoryStore()


# ============================================================
# 2. USER
# ============================================================

assistant_id = "my-assistant"
user_id = "user-001"


# ============================================================
# 3. USER-SCOPED NAMESPACE
# ============================================================

user_namespace = (
    assistant_id,
    user_id,
)


# ============================================================
# 4. BACKEND
# ============================================================

backend = CompositeBackend(
    default=StateBackend(),

    routes={
        "/memories/": StoreBackend(
            namespace=user_namespace,
        ),

        "/skills/": StoreBackend(
            namespace=(assistant_id,),
        ),
    },
)


# ============================================================
# 5. CREATE AGENT
# ============================================================

agent = create_deep_agent(
    model="openrouter:openai/gpt-4o-mini",

    memory=[
        r"projects\agent.md"
    ],

    skills=[
        r"skills\python\SKILL.md"
    ],

    backend=backend,

    store=store,
)


# ============================================================
# 6. SAVE USER MEMORY
# ============================================================

store.put(
    user_namespace,
    "preferences",
    {
        "name": "Nayyab",
        "response_style": "detailed",
        "preferred_language": "English",
        "framework": "LangChain",
    },
)


# ============================================================
# 7. READ USER MEMORY
# ============================================================

memory = store.get(
    user_namespace,
    "preferences",
)

print("USER MEMORY:")
print(memory.value)


USER MEMORY:
{'name': 'Nayyab', 'response_style': 'detailed', 'preferred_language': 'English', 'framework': 'LangChain'}


In [22]:
memory = store.get(
    user_namespace,
    "preferences",
)

preferences = memory.value if memory else {}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": f"""
Here are the user's saved preferences:

{preferences}

Now answer the user's question:
What are my preferences?
"""
            }
        ]
    },
    config=config,
)

print(result["messages"][-1].content)


Your saved preferences are as follows:

- **Name**: Nayyab
- **Response Style**: Detailed
- **Preferred Language**: English
- **Framework**: LangChain


# Long Term Memory
## 1. Semantic Memory

In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

semantic_namespace = ("assistant", "user-001")

store.put(
    semantic_namespace,
    "user_profile",
    {
        "name": "Nayyab",
        "preferred_framework": "LangChain",
        "response_style": "detailed",
        "experience": "AI Engineer"
    }
)

# Retrieve semantic memory
memory = store.get(
    semantic_namespace,
    "user_profile"
)

print(memory.value)


{'name': 'Nayyab', 'preferred_framework': 'LangChain', 'response_style': 'detailed', 'experience': 'AI Engineer'}


## 2. Episodic Memory

In [ ]:
from datetime import datetime

episodic_namespace = ("assistant", "user-001", "episodes")

store.put(
    episodic_namespace,
    "episode-001",
    {
        "timestamp": datetime.now().isoformat(),
        "event": "Learned about Deep Agents memory",
        "topic": "StateBackend vs StoreBackend",
        "outcome": "Understood that StoreBackend is used for persistent memory",
        "importance": "high"
    }
)

# Retrieve the episode
episode = store.get(
    episodic_namespace,
    "episode-001"
)

print(episode.value)


{'timestamp': '2026-10-06T16:53:50.965899', 'event': 'Learned about Deep Agents memory', 'topic': 'StateBackend vs StoreBackend', 'outcome': 'Understood that StoreBackend is used for persistent memory', 'importance': 'high'}


## 3. Procedural Memory

In [ ]:
procedural_namespace = (
    "assistant",
    "procedures"
)

store.put(
    procedural_namespace,
    "python_coding_procedure",
    {
        "name": "Python coding workflow",
        "steps": [
            "Understand the requirement",
            "Check existing code",
            "Implement the solution",
            "Run tests",
            "Handle errors",
            "Explain the final implementation"
        ],
        "rules": [
            "Use Python 3.12",
            "Prefer type hints",
            "Do not use unnecessary dependencies"
        ]
    }
)

procedure = store.get(
    procedural_namespace,
    "python_coding_procedure"
)

print(procedure.value)


{'name': 'Python coding workflow', 'steps': ['Understand the requirement', 'Check existing code', 'Implement the solution', 'Run tests', 'Handle errors', 'Explain the final implementation'], 'rules': ['Use Python 3.12', 'Prefer type hints', 'Do not use unnecessary dependencies']}
